# FaceFusion 3.3.0 - Google Colab

This notebook allows you to run FaceFusion in Google Colab for face swapping and video generation.

**Requirements:**
- GPU Runtime (T4 or better recommended)
- Google Drive (optional, for saving outputs)

**Usage:**
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run cells in order
3. Upload your source and target media when prompted

## 1. Check GPU Availability

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. Install System Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y ffmpeg libsm6 libxext6 libxrender-dev libgomp1

## 3. Clone FaceFusion Repository

In [ ]:
import os

# Remove existing directory if present
if os.path.exists('facefusion'):
    !rm -rf facefusion

# Clone the repository
!git clone https://github.com/facefusion/facefusion.git
%cd facefusion

# Checkout version 3.3.0
!git checkout 3.3.0 || echo "Using main branch"

## 4. Install Python Dependencies

In [ ]:
# Install requirements
!pip install -q -r requirements.txt

# Install additional dependencies for Colab
!pip install -q gradio

## 5. Mount Google Drive (Optional)
Mount your Google Drive to save outputs permanently

In [ ]:
# Uncomment the lines below to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Create output directory in Drive
# !mkdir -p /content/drive/MyDrive/FaceFusion_Outputs

## 6. Upload Source and Target Files

In [ ]:
from google.colab import files
import shutil

# Create directories
!mkdir -p inputs outputs

print("Upload your SOURCE image (face to swap FROM):")
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f'inputs/{filename}')
    source_path = f'inputs/{filename}'
    print(f"Source saved: {source_path}")

print("\nUpload your TARGET image/video (face to swap TO):")
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f'inputs/{filename}')
    target_path = f'inputs/{filename}'
    print(f"Target saved: {target_path}")

## 7. Run FaceFusion (Command Line)
Modify the parameters below as needed

In [ ]:
# Basic face swap command
# Replace SOURCE_IMAGE and TARGET_FILE with your uploaded filenames

SOURCE_IMAGE = "source.jpg"  # Change this to your source image filename
TARGET_FILE = "target.mp4"   # Change this to your target image/video filename
OUTPUT_FILE = "outputs/result.mp4"

# Run FaceFusion
!python run.py \
  --source inputs/{SOURCE_IMAGE} \
  --target inputs/{TARGET_FILE} \
  --output {OUTPUT_FILE} \
  --execution-providers cuda \
  --headless

## 8. Alternative: Launch Web UI
Run FaceFusion with Gradio web interface

In [ ]:
# Launch the web UI (with public URL)
!python run.py --ui-name gradio --execution-providers cuda

## 9. Download Output Files

In [ ]:
from google.colab import files
import os

# List all output files
print("Available output files:")
output_files = [f for f in os.listdir('outputs') if os.path.isfile(f'outputs/{f}')]
for i, f in enumerate(output_files):
    print(f"{i+1}. {f}")

# Download the result
if output_files:
    files.download(f'outputs/{output_files[0]}')
else:
    print("No output files found. Make sure the processing completed successfully.")

## 10. Advanced Options

Additional command line options you can use:

```bash
# Face enhancer
--face-enhancer gfpgan_1.4

# Frame enhancer
--frame-enhancer real_esrgan_x2plus

# Face analyzer
--face-analyzer-model buffalo_l

# Processing speed vs quality
--execution-thread-count 4
--execution-queue-count 2

# Skip NSFW checks (use with caution)
--skip-download
```

## 11. Troubleshooting

If you encounter issues:

1. **Out of Memory**: Reduce video resolution or use a smaller model
2. **Slow Processing**: Make sure GPU is enabled in Runtime settings
3. **Model Download Issues**: Check your internet connection
4. **CUDA Errors**: Restart runtime and try again

For more help, visit: https://github.com/facefusion/facefusion